In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# LOAD ALL TABLES
# ============================================================
match_stats = pd.read_csv('MatchStatsTbl.csv')
match = pd.read_csv('MatchTbl.csv')
team_match = pd.read_csv('TeamMatchTbl.csv')
summoner_match = pd.read_csv('SummonerMatchTbl.csv')
rank = pd.read_csv('RankTbl.csv')
champion = pd.read_csv('ChampionTbl.csv')
item = pd.read_csv('ItemTbl.csv')

# ============================================================
# REAPPLY CLASSIC FILTER + DURATION FILTER
# Same cleaning steps from EDA
# Only includestandard 5v5 games between 10-60 mins for analysis
# ============================================================
classic_matches = match[match['QueueType'] == 'CLASSIC']
classic_matches = classic_matches[
    (classic_matches['GameDuration'] >= 600) &
    (classic_matches['GameDuration'] <= 3600)
]

print("CLASSIC matches loaded:", len(classic_matches))
print("MatchStatsTbl rows:", len(match_stats))

CLASSIC matches loaded: 186641
MatchStatsTbl rows: 732308


In [ ]:
# ============================================================
# STEP 1: MERGE TABLES
# We need to connect MatchStatsTbl (player performance)
# with SummonerMatchTbl (links player to match)
# with MatchTbl (gives us rank and game duration)
#
# The join chain is:
# MatchStatsTbl → SummonerMatchTbl → MatchTbl → RankTbl
# ============================================================

# First join: connect player stats to summoner-match link table
df = match_stats.merge(
    summoner_match,
    left_on = 'SummonerMatchFk',
    right_on = 'SummonerMatchId',
    how ='inner'

)

# Second join: connect to match info (rank, duration, queue type)
df = df.merge(
    classic_matches[['MatchId','Patch','RankFk','GameDuration']],
    left_on = 'MatchFk',
    right_on = 'MatchId',
    how = 'inner'



)

# Third join: convert rank number to readable rank name
df = df.merge(
    rank,
    left_on='RankFk',
    right_on='RankId',
    how='left'
)

print("Merged dataframe shape:", df.shape)
print("\nColumns available:")
print(df.columns.tolist())

Merged dataframe shape: (433537, 41)

Columns available:
['MatchStatsId', 'SummonerMatchFk', 'MinionsKilled', 'DmgDealt', 'DmgTaken', 'TurretDmgDealt', 'TotalGold', 'Lane', 'Win', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'kills', 'deaths', 'assists', 'PrimaryKeyStone', 'PrimarySlot1', 'PrimarySlot2', 'PrimarySlot3', 'SecondarySlot1', 'SecondarySlot2', 'SummonerSpell1', 'SummonerSpell2', 'CurrentMasteryPoints', 'EnemyChampionFk', 'DragonKills', 'BaronKills', 'visionScore', 'SummonerMatchId', 'SummonerFk', 'MatchFk', 'ChampionFk', 'MatchId', 'Patch', 'RankFk', 'GameDuration', 'RankId', 'RankName']


In [ ]:
# ============================================================
# STEP 2: CLEAN LANE COLUMN
# We saw in EDA that Lane has nulls, NONE, and SUPPORT
# We standardise these so lane is usable as a feature
# SUPPORT gets remapped to UTILITY (same role, inconsistent labelling)
# NONE and nulls get labelled UNKNOWN — we keep them rather
# than dropping 177K rows which would lose too much data
# ============================================================

df['Lane'] = df['Lane'].fillna('UNKNOWN')
df['Lane'] = df['Lane'].replace('SUPPORT', 'UTILITY')
df['Lane'] = df['Lane'].replace('NONE', 'UNKNOWN')

print("Lane distribution after cleaning:")
print(df['Lane'].value_counts())

Lane distribution after cleaning:
Lane
BOTTOM     106734
JUNGLE      88562
MIDDLE      85286
TOP         80408
UTILITY     52910
UNKNOWN     19637
Name: count, dtype: int64


In [ ]:
# ============================================================
# STEP 3: ENGINEER PLAYER LEVEL FEATURES
# We create new columns that better capture player performance
# than the raw numbers alone
# ============================================================

# KDA Ratio: measures combat effectiveness
# Formula: (kills + assists) / deaths
# We use max(deaths, 1) to avoid dividing by zero
# e.g. 10 kills, 5 assists, 2 deaths = (10+5)/2 = 7.5 KDA

df['kda_ratio'] = (df['kills'] + df['assists']) / df['deaths'].clip(lower=1)

# Gold Per Minute: measures how efficiently a player farmed
# Total gold earned divided by game duration in minutes
# e.g. 12000 gold in 30 mins = 400 gold/min
df['gold_per_min'] = df['TotalGold'] / (df['GameDuration'] / 60)

# Damage Per Minute: normalises damage by game length
# A player dealing 50K damage in 20 mins > 50K in 40 mins
df['dmg_per_min'] = df['DmgDealt'] / (df['GameDuration'] / 60)

# Kill Participation: what % of team kills did this player contribute to
# (kills + assists) / team total kills
# First we need total team kills — we get this from TeamMatchTbl later
# For now we create a simpler version using DragonKills + BaronKills
# as a proxy for objective participation
df['objective_participation'] = df['DragonKills'] + df['BaronKills']

# CS Per Minute: minions killed per minute = farming efficiency
df['cs_per_min'] = df['MinionsKilled'] / (df['GameDuration'] / 60)

print("New features created:")
print(df[['kda_ratio', 'gold_per_min', 'dmg_per_min',
          'objective_participation', 'cs_per_min']].describe().round(2))

New features created:
       kda_ratio  gold_per_min  dmg_per_min  objective_participation  \
count  433537.00     433537.00    433537.00                433537.00   
mean        3.49        400.07       713.36                     0.45   
std         3.84         91.38       345.45                     1.03   
min         0.00        136.51         0.00                     0.00   
25%         1.25        332.15       455.52                     0.00   
50%         2.25        394.20       669.56                     0.00   
75%         4.17        459.11       913.89                     0.00   
max        46.00        964.59      3841.99                     9.00   

       cs_per_min  
count   433537.00  
mean         4.44  
std          2.98  
min          0.00  
25%          1.11  
50%          5.45  
75%          7.01  
max         12.89  


In [ ]:
# ============================================================
# STEP 4: CAP OUTLIERS
# Extreme values can distort our model
# We cap at the 99th percentile — meaning we keep 99% of data
# and only flatten the top 1% of extreme values
# This is called "winsorizing" in statistics
# ============================================================

# Calculate 99th percentile caps for each feature
cap_kda = df['kda_ratio'].quantile(0.99)
cap_gold = df['gold_per_min'].quantile(0.99)
cap_dmg = df['dmg_per_min'].quantile(0.99)
cap_cs = df['cs_per_min'].quantile(0.99)
cap_vision = df['visionScore'].quantile(0.99)

print("Caps being applied:")
print(f"  kda_ratio capped at:    {cap_kda:.2f}")
print(f"  gold_per_min capped at: {cap_gold:.2f}")
print(f"  dmg_per_min capped at:  {cap_dmg:.2f}")
print(f"  cs_per_min capped at:   {cap_cs:.2f}")
print(f"  visionScore capped at:  {cap_vision:.2f}")

# Apply the caps using clip — values above cap get set to the cap value
df['kda_ratio'] = df['kda_ratio'].clip(upper=cap_kda)
df['gold_per_min'] = df['gold_per_min'].clip(upper=cap_gold)
df['dmg_per_min'] = df['dmg_per_min'].clip(upper=cap_dmg)
df['cs_per_min'] = df['cs_per_min'].clip(upper=cap_cs)
df['visionScore'] = df['visionScore'].clip(upper=cap_vision)

# Quick check that capping worked - using round() directly on the number
print("\nAfter capping — kda_ratio max:", round(df['kda_ratio'].max(), 2))
print("After capping — dmg_per_min max:", round(df['dmg_per_min'].max(), 2))

Caps being applied:
  kda_ratio capped at:    20.00
  gold_per_min capped at: 639.08
  dmg_per_min capped at:  1725.02
  cs_per_min capped at:   9.40
  visionScore capped at:  120.00

After capping — kda_ratio max: 20.0
After capping — dmg_per_min max: 1725.02


In [ ]:
# ============================================================
# STEP 5: SELECT WORKING COLUMNS
# We keep the in-game features (needed to BUILD historical
# averages), plus IDs needed for chronological ordering.
# Note: we do NOT save these in-game stats as model inputs —
# they're only the raw material for the historical features.
# ============================================================
features = [
    'Win',                       # target
    'kda_ratio', 'gold_per_min', 'dmg_per_min',
    'cs_per_min', 'visionScore', 'objective_participation',
    'Lane', 'RankName', 'GameDuration',
    'SummonerFk', 'MatchFk'      # IDs for ordering & grouping
]
df = df[features].copy()
print("Working set shape:", df.shape)

Working set shape: (433537, 12)


In [ ]:
# ============================================================
# STEP 6: ORDER MATCHES CHRONOLOGICALLY PER PLAYER
# Riot match IDs increase over time (higher = more recent).
# We extract the numeric part and sort each player's matches
# oldest → newest so historical averages are built correctly.
# ============================================================
df['match_num'] = df['MatchFk'].str.extract(r'_(\d+)').astype(float)
df = df.sort_values(['SummonerFk', 'match_num']).reset_index(drop=True)
print("Sorted. Sample:")
print(df[['SummonerFk', 'match_num', 'kda_ratio']].head())

Sorted. Sample:
   SummonerFk     match_num  kda_ratio
0           1  7.548453e+09       6.25
1           1  7.555468e+09      13.00
2           1  7.556674e+09      13.50
3           1  7.556940e+09       7.00
4           1  7.563606e+09      16.00


In [ ]:
# ============================================================
# STEP 7: BUILD HISTORICAL FEATURES
# For each behavioral feature, compute the player's running
# average from PRIOR matches only.
#   shift(1)           → exclude the current match
#   expanding().mean() → average of all prior matches
# This converts in-game stats into pre-game knowledge.
# ============================================================
behavioral_features = ['kda_ratio', 'gold_per_min', 'dmg_per_min',
                       'cs_per_min', 'visionScore', 'objective_participation']

grouped = df.groupby('SummonerFk')
for feat in behavioral_features:
    shifted = grouped[feat].shift(1)
    df[f'hist_{feat}'] = shifted.groupby(df['SummonerFk']).expanding().mean().reset_index(level=0, drop=True)

print("Historical features built.")
print(df[df['SummonerFk'] == 2][['match_num', 'kda_ratio', 'hist_kda_ratio']].head(6))

Historical features built.
       match_num  kda_ratio  hist_kda_ratio
10  7.516496e+09   2.000000             NaN
11  7.517661e+09   0.833333        2.000000
12  7.517735e+09   3.333333        1.416667
13  7.519172e+09   1.416667        2.055556
14  7.519236e+09   1.000000        1.895833
15  7.519266e+09   4.750000        1.716667


In [ ]:
# ============================================================
# STEP 8: FINALIZE MODELING DATASET
# Drop each player's first match (NaN history — no prior games).
# Keep hist_ features + context (Lane, Rank) + IDs + target.
# Save for notebooks 03 and 04.
# ============================================================
hist_features = [f'hist_{f}' for f in behavioral_features]

# Keep only what downstream notebooks need
keep_cols = ['Win'] + hist_features + ['Lane', 'RankName', 'GameDuration', 'SummonerFk', 'MatchFk', 'match_num']
df_final = df[keep_cols].dropna(subset=hist_features).copy()

print("Rows before drop:", len(df))
print("Rows after dropping first matches:", len(df_final))
print("Win rate:", round(df_final['Win'].mean(), 3))

df_final.to_csv('df_model_historical.csv', index=False)
print("\nSaved df_model_historical.csv")
print("Columns:", df_final.columns.tolist())

Rows before drop: 433537
Rows after dropping first matches: 242757
Win rate: 0.503

Saved df_model_historical.csv
Columns: ['Win', 'hist_kda_ratio', 'hist_gold_per_min', 'hist_dmg_per_min', 'hist_cs_per_min', 'hist_visionScore', 'hist_objective_participation', 'Lane', 'RankName', 'GameDuration', 'SummonerFk', 'MatchFk', 'match_num']


In [ ]:
print(df_final.head())

   Win  hist_kda_ratio  hist_gold_per_min  hist_dmg_per_min  hist_cs_per_min  \
1    1        6.250000         386.496956        593.458771         1.627006   
2    1        9.625000         320.930567        384.035845         1.212037   
3    0       10.916667         320.886886        358.255400         1.213753   
4    1        9.937500         300.464159        293.189037         1.161571   
5    0       11.150000         302.135270        287.754862         1.090087   

   hist_visionScore  hist_objective_participation    Lane RankName  \
1              80.0                           0.0  BOTTOM  Diamond   
2              89.5                           0.0  BOTTOM  Diamond   
3              86.0                           0.0  BOTTOM  Diamond   
4              81.0                           0.0  BOTTOM  Diamond   
5              81.4                           0.0  BOTTOM  Diamond   

   GameDuration  SummonerFk          MatchFk     match_num  
1          2183           1  EUW1_755